# Lesson 9 : Hosted Agents in Microsoft Foundry

When you deploy and run your generated agent's code for production, you can, of course, host your agent by yourself.  
However, if you're working in Microsoft Foundry, you can host your agent as a hosted agent in Microsoft Foundry.  
By using hosted agents in Microsoft Foundry, you can handle your agents without having to build various management functionalities - such as, start/stop, upgrading/versioning, monitoring, horizontal scaling, etc.

> Note : The code built by Agent Framework, LangChain, and custom code can be run as a hosted agent. (Workflow agents can also work as hosted agents.)

In this exercise, we configure and deploy our agent (which is built in Lesson 4) as a hosted agent in Microsoft Foundry.

Here I'll show you the outline of steps to run hosted agent, but please see [official document](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/hosted-agents) for more details.

> Note : I note that currently (Jan 2026) this feature is in preview and there's several restrictions as the time of writing, and some additional tasks to be done.

## 1. Prepare environment

To start this exercise, you should install ```azd``` command (Azure Developer CLI) on your working environment.  
After installation, login to Azure in ```azd``` by running ```azd auth login --use-device-code```.

> Note : Here we use ```azd``` command to configure infrastructure, but you can also manually provision Azure infrastructure and deploy your hosted agent with Python SDK.

Next, install docker, because hosted agents run on container app as a base.  
In my case, I have used the following command to install docker in Ubuntu 24.04, and logout/login after installation.

```
sudo apt-get -y update
sudo apt-get -y install apt-transport-https ca-certificates curl software-properties-common
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get -y update
apt-cache policy docker-ce
sudo apt-get -y install docker-ce
sudo usermod -aG docker $USER　 # Logout and Login to take effect !
```

## 2. Prepare resources in Azure

Currently (Jan 2026), hosted agents are supported on Microsoft Foundry in North Central US region only.  
So please create another Foundry resource and Foundry project in North Central US region.

> Note : In this exercise, we prepare Foundry resource in advance, but you can also create Foundry resource in the following provisioning phase, without creating Foundry resources manually by yourself.

In Foundry Portal, deploy an Azure OpenAI model which is supported in Azure OpenAI Responses API. (See [here](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?view=foundry&tabs=python-key#model-support) for the supported models.)  
In the following setting, we assume we have deployed **gpt-5**.

In hosted agents, other resources (such as, container registry, etc) are also required, but these are automatically created by running the following provisioning steps, and there's no other resources to be prepared here.

## 3. Prepare files

Hosted agents use docker container as its base technology.  
In order for provisioning as a hosted agent, you should prepare the following files in this example.

- Python source code (main.py)
- Python package list (requirements.txt)
- Docker file (Dockerfile)
- Agent configuration (agent.yaml)

In this example, we save these files in ```hosted_agent_work/script``` folder, and we create this folder as follows.

In [1]:
import os
script_folder = path = os.path.join("hosted_agent_work", "script")
os.makedirs(script_folder, exist_ok=True)

First we prepare Python source code (main.py).

In this example, we change code in Lesson 4 as follows in order to deploy this agent as a hosted agent in Microsoft Foundry.

- The agent is abstracted by adapter function, ```from_agent_framework()```.
- To get credential, I have changed ```AzureCliCredential``` to ```DefaultAzureCredential```. (Because the credential is not set by ```az login``` in hosted agent.)

The ```%%writefile``` directive (first line in the following code) in Jupyter notebook cell indicates that this code is saved as file, not executed here.

In [2]:
%%writefile hosted_agent_work/script/main.py
from agent_framework.azure import AzureAIClient
from agent_framework import ChatAgent, HostedWebSearchTool
from azure.identity import DefaultAzureCredential
from azure.ai.agentserver.agentframework import from_agent_framework

def create_agent():
    credential = DefaultAzureCredential()
    client = AzureAIClient(credential=credential)
    agent = ChatAgent(
        name="WeatherHostedAgent",
        chat_client=client,
        instructions="You are an agent about weather information.",  # "あなたは、気象情報に関する Agent です。"
        tools=[HostedWebSearchTool()]
    )
    return agent

if __name__ == "__main__":
    from_agent_framework(lambda _: create_agent()).run()

Writing hosted_agent_work/script/main.py


Next we prepare requirements.txt, a list of Python packages, which are installed on docker image generation.

To use adapter for hosted agent (i.e., ```from_agent_framework()``` in above code), we should install Python package ```azure-ai-agentserver-agentframework```.

Currently (Jan 2026) it has conflict between ```azure-ai-agentserver-agentframework``` and ```agent-framework```, and here I use old version of ```agent-framework``` packages by specifying versions. (See [here](https://github.com/Azure/azure-sdk-for-python/issues/44695) for this issue.)

In [3]:
%%writefile hosted_agent_work/script/requirements.txt
azure-ai-agentserver-agentframework==1.0.0b7
agent-framework-azure-ai==1.0.0b260107
agent-framework==1.0.0b260107
agent-framework-core==1.0.0b260107

Writing hosted_agent_work/script/requirements.txt


Now we create docker file, ```Dockerfile```.  
The above ```from_agent_framework()``` automatically creates a REST endpoint by port 8088 and we then expose this port in docker image generation.

In [4]:
%%writefile hosted_agent_work/script/Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY . user_agent/
WORKDIR /app/user_agent

RUN if [ -f requirements.txt ]; then \
        pip install -r requirements.txt; \
    else \
        echo "No requirements.txt found"; \
    fi

EXPOSE 8088

CMD ["python", "main.py"]

Writing hosted_agent_work/script/Dockerfile


The file, agent.yaml, includes configuration settings (such as, agent name, environment variables, etc), which are used in the following provisioning phase by ```azd``` command execution.

As you saw above, we use ```AzureAIClient``` in this example, and this client then requires ```AZURE_AI_PROJECT_ENDPOINT``` and ```AZURE_AI_MODEL_DEPLOYMENT_NAME```.  
In this example, therefore, we set these variables as environment variables as follows. (You can also pass these variables by ```.env``` file, instead.)

> Note : In the following yaml configuration, you can see ```${AZURE_AI_PROJECT_ENDPOINT}```, but you don't need to set this environment variable manually by yourself. This will be automatically set up in the following "```azd ai agent init --project-id [PROJECT-RESOURCE-ID]```" command.

**Please replace the following "```gpt-5```" with your setting.**

In [5]:
%%writefile hosted_agent_work/script/agent.yaml
name: hosted-test-agent01
description: This is a hosted agent for demo.
metadata:
  authors:
    - Workshop demo
template:
  name: hosted-test-agent01
  kind: hosted
  protocols:
    - protocol: responses
      version: v1
  environment_variables:
    - name: AZURE_AI_PROJECT_ENDPOINT
      value: ${AZURE_AI_PROJECT_ENDPOINT}
    - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
      value: "{{chat}}"
resources:
  - kind: model
    id: gpt-5
    name: chat

Writing hosted_agent_work/script/agent.yaml


## 4. Provision, deploy, and run

Now the assets for provisioning are all ready.  
We now start to complete configurations, provision Azure infrastructure, and deploy your hosted agent, with ```azd``` commands.

**All the following settings should be performed in console (terminal)**, not in Jupyter notebook. (Because Jupyter notebook cannot handle the interactive session.)

Before starting, let's change your working directory to subfolder ```hosted_agent_work```. (Because the execution commands automatically detect the files in your directory.)

```bash
cd hosted_agent_work
```

---

First, run the following command to prepare infrastructure setting as bicep configuration.  
Required configurations are downloaded from GitHub repository and setup in "```infra```" subfolder.

```bash
azd init -t https://github.com/Azure-Samples/azd-ai-starter-basic
```

During running this command, a unique environment name will be asked, but please set arbitrary name. (See the following note.)

> Note : This unique environment name is used in resource group name, repository name, and a tag in Foundry resource, etc. In this example, we have already prepared Foundry resource and its resource group.

---

Next we configure this setting to use our prepared Foundry resource, by running the following command.  
In this command, **please change the following placeholders, before running**. To get string in ```--project-id``` option, go to foundry project resource on [Azure Portal](https://portal.azure.com/), open "Resource Management" - "properties" in left-side navigation, and you then find it in "Resource ID".  
By running this command, the environment variables associated with Foundry resource (such as, ```AZURE_AI_PROJECT_ENDPOINT```, ```AZURE_OPENAI_ENDPOINT```, etc) are automatically retrieved and set in configuration.

```bash
azd ai agent init --project-id /subscriptions/[SUBSCRIPTION-ID]/resourceGroups/[RESOURCE-GROUP-NAME]/providers/Microsoft.CognitiveServices/accounts/[FOUNDRY-RESOURCE-NAME]/projects/[PROJECT-NAME]
```

During running this command, you will be asked to specify existing Azure Container Registry (ACR) server name and Application Insights connection name. But please set blank, and it will then automatically create these resources in provisioning steps.

> Note : If you have already connected to these resources (app insights, container registry) in your Foundry resource, do not set blank, because multiple connection to the same category are not allowed in Microsoft Foundry. (The error will be thrown.)

> Note : When you skip this step, entire resources (including resource group, Foundry resource, and Foundry project resource) are automatically generated in the following provisioning phase. In this method, the unique environment name in the setting (see above) is used as the name of resource group and Foundry project.  
> Especially in development, we want to quickly build and drop the environment, and this method could be helpful. (See the following note about ```azd up``` and ```azd down```.)

---

Next run the following command to complete the agent settings.

```bash
azd ai agent init -m script/agent.yaml
```

During running this command, you will be asked for the following settings, but here we apply the default settings as follows.  
These provisioning definitions are then written in azure.yaml.

- container memory allocation : 2Gi
- container CPU allocation : 1
- container minimum number of replicas : 1
- container maximum number of replicas : 3

---

Now let's provision the infrastructure by running the following command.  
After running this command, all required resources - Application Insights, Log Analytics workspace, and Azure Container Registry (ACR) - are provisioned in the same resource group as your Foundry resource. (Also, the container image is built and registered in ACR.)

```bash
azd provision
```

Currently (Jan 2026), you should additionally set the following ```AcrPull``` permission after the above provisioning has completed. (See [here](https://github.com/microsoft-foundry/foundry-samples/issues/454) for this issue.)

- Go to container registry resource.
- Go to "Access control (IAM)".
- Add "```AcrPull```" role to managed identity of Foundry resource (not Foundry project resource).

---

Now all Azure resources are ready.  
Finally we deploy and start our hosted agent in Microsoft Foundry by running the following command.

```bash
azd deploy hosted-test-agent01
```

> Note :  By running ```azd up```, provisioning (```azd provision```) and deployment (```azd deploy```) are both performed.  
> On the other hand, ```azd down``` cleans up all resources in resource group.  
> By using ```azd up``` / ```azd down```, you can quickly build all resources and clean up all resources.

After the hosted agent is successfully deployed, you can see your agent running in Foundry Portal UI. (The hosted agent is labeled "```hosted```" as type.)

---

You can start / stop your agent in Foundry Portal.  
Or you can also use "```az cognitiveservices agent```" CLI commands to manage your agent as follows. (Run ```az cognitiveservices agent --help``` or see [official document](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/hosted-agents#manage-hosted-agents) for other management commands.)

```bash
# start agent
az cognitiveservices agent start \
  --account-name [FOUNDRY-RESOURCE-NAME] \
  --project-name [FOUNDRY-PROJECT-NAME] \
  --name [AGENT-NAME] \
  --agent-version [AGENT-VERSION]
# stop agent
az cognitiveservices agent stop \
  --account-name [FOUNDRY-RESOURCE-NAME] \
  --project-name [FOUNDRY-PROJECT-NAME] \
  --name [AGENT-NAME] \
  --agent-version [AGENT-VERSION]
```

> Note : Before deploying as hosted agent, I recommend you to run the container in local docker runtime. (Once deployed, the error tracking will become more difficult.)

## 5. Consume your hosted agent

To consume your hosted agent, you can use Playground in Foundry Portal UI, or invoke API.

The API interface exposed by the hosted agent is a REST API that is compatible with OpenAI Responses API. (See "```protocol: responses```" in above agent.yaml setting.)  
You can then use Azure AI Projects SDK (```azure-ai-projects```), OpenAI SDK, or raw REST to consume your hosted agent with API. (See [official document](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/hosted-agents#invoke-hosted-agents) for details.)

Once it's deployed as hosted agent, your agent can also be used in Microsoft ecosystem - such as, publishing to Microsoft 365 Copilot, using in multi-agent workflows on Foundry, etc.